# 09 Unitary Dilation And Post-Selection

Construct one-ancilla unitary dilation for a diagonal non-unitary operator and simulate ancilla post-selection.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
n_qubits = 4
_, _, dx = make_log_price_grid(n_qubits, center_price=24000, x_width=config["quantum"]["x_width"])
p = momentum_eigenvalues(2 ** n_qubits, dx)
O, scale = non_unitary_anti_hermitian_diagonal(p, tau=30/365, sigma=0.18, r=0.065)
U = dilation_matrix_from_diagonal(O)
unitarity_error = dilation_unitarity_error(U)
assert unitarity_error < 1e-10
state = np.ones(2 ** n_qubits, dtype=complex) / math.sqrt(2 ** n_qubits)
post = apply_dilation_and_postselect(O, state)
assert 0 <= post["probability"] <= 1
assert abs(np.linalg.norm(post["post_selected_state"]) - 1.0) < 1e-12
print("VALIDATION PASSED: unitary dilation matrix, post-selection probability, and normalized recovered state")
plt.figure()
plt.plot(np.abs(O), marker="o")
plt.title("Diagonal contraction values for anti-Hermitian evolution")
plt.xlabel("Momentum index")
plt.ylabel("|O_k|")
save_current_figure("09_nonunitary_operator_diagonal.png")
summary = {"unitarity_error": unitarity_error, "post_selection_probability": post["probability"], "normalization_scale": scale}
save_output(summary, "09_unitary_dilation_post_selection.json")
summary
